# CNN a klasyczne filtry obrazu

Celem ćwiczenia jest zbadanie związku między **konwolucyjnymi sieciami neuronowymi (CNN)**
a **klasycznymi filtrami obrazu**.

Nauczymy prostą sieć CNN złożoną z jednego filtra konwolucyjnego, by wykrywała krawędzie
tak jak operator Sobela. Następnie sprawdzimy, czy wyuczone wagi odpowiadają
teoretycznemu jądru Sobela.

**Plan ćwiczenia:**
1. Wczytanie obrazu i wykrycie krawędzi operatorem Sobela
2. Przygotowanie danych treningowych
3. Uczenie jednowarstwowej sieci CNN
4. Porównanie wag sieci z jądrem Sobela
5. Powtórzenie eksperymentu z własnym filtrem (`filter2D`)

## 1. Importy i wczytanie obrazu

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import keras
from keras import layers

def imshow(image):
    plt.imshow(image, cmap="gray")
    plt.show()

In [ ]:
# TODO: wczytaj obraz w skali szarości (cv.imread() z odpowiednią flagą)
# img = 

# Wyświetlenie wczytanego obrazu oraz jego rozmiaru
imshow(img)

print(f"Rozmiar obrazu: {img.shape}, typ: {img.dtype}")

## 2. Detekcja krawędzi operatorem Sobela

Operator Sobela oblicza numeryczną pochodną obrazu w kierunku poziomym (X) lub pionowym (Y).
Dla kierunku X stosuje się jądro:

$$K_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$$

Wartości dodatnie odpowiadają przejściom ciemny→jasny, ujemne — jasny→ciemny.
Wynik może zawierać wartości ujemne — dlatego należy użyć `cv.CV_32F`.

In [ ]:
# Teoretyczne jądro Sobela w kierunku X — do porównania z wagami sieci
sobel_kernel_x = np.array([
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1]
], dtype=np.float32)

print("Jądro Sobela (oś X):")
print(sobel_kernel_x)

In [ ]:
# TODO: wykryj krawędzie operatorem Sobela (funkcja cv.Sobel)
# parametry ustawić w taki sposób, żeby: była obliczana pierwsza pochodna w kierynku X
# jako typ należy ustawić cv.CV_32F,
# rozmiar dobrać, żeby był taki sam jak rozmiar filtru w sieci neuronowej
# WAŻNE!!! borderType ustawić, żeby odpowiadał paddingowi w sieci neuronowej

# edges = 

# Normalizacja |edges| do [0, 1] wyłącznie na potrzeby wizualizacji
edges_vis = cv.normalize(np.abs(edges), None, 0, 1, cv.NORM_MINMAX)

imshow(edges_vis)

## 3. Przygotowanie danych treningowych

Sieć CNN oczekuje danych w formacie `(batch, height, width, channels)`.
Mamy jeden obraz, więc:
- `train_x` — obraz wejściowy: kształt `(1, H, W, 1)`, wartości `float32`
- `train_y` — wynik Sobela: kształt `(1, H, W, 1)`, wartości `float32` (mogą być ujemne!)

In [ ]:
# Sieć CNN oczekuje danych w formacie (batch, height, width, channels)
# np.expand_dims dodaje brakujące wymiary: batch (oś 0) i kanał (oś -1)
train_x = np.expand_dims(img, axis=(0, -1)).astype(np.float32)
train_y = np.expand_dims(edges, axis=(0, -1))

print(f"train_x: kształt={train_x.shape}")
print(f"train_y: kształt={train_y.shape}")

## 4. Model sieci neuronowej

Sieć składa się z **jednej warstwy konwolucyjnej** z:
- jednym filtrem
- rozmiarem kernela takim samym jaki został wybrany w Sobelu
- padding ustawiony na odpowiadający borderType w funkcji Sobel.
- **brak funkcji aktywacji** — sieć uczy się liniowego przekształcenia.

In [ ]:
model = keras.Sequential([
    # TODO: uzupełnić sieć neuronową
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.1), loss="mse")
model.summary()

In [ ]:
history = model.fit(train_x, train_y, epochs=500)

## 5. Porównanie wag z jądrem Sobela

Po treningu wagi warstwy konwolucyjnej powinny być zbliżone do jądra Sobela X.

In [ ]:
weights, biases = model.layers[0].get_weights()
learned_kernel = weights[:, :, 0, 0]   # kształt (3, 3)

print("Wyuczone wagi (jądro):")
print(np.round(learned_kernel, 3))
print(f"Wyuczony bias: {biases[0]:.6f}")

# Jądro Sobela przeskalowane do porównania
print("Jądro Sobela:")
print(np.round(sobel_kernel_x, 3))

## 6. Eksperyment z własnym filtrem (`filter2D`)

Powtórzymy eksperyment z sekcji 2–5, ale tym razem zamiast operatora Sobela
użyjemy **dowolnego filtru zdefiniowanego ręcznie** i funkcji `cv.filter2D`.

Czy sieć będzie w stanie odtworzyć ten filtr?

In [ ]:
# TODO Powtórka eksperymentu, ale korzystając z ustawionego przez siebie kernela i funkcji filter2D